# 1. Experiment Information

## EXP003 — Baseline HT-Demucs Evaluation on BAUL GEET

### Experiment Aim

To evaluate the performance of the pretrained HT-Demucs source separation model for two-stem vocal and accompaniment separation on selected Baul Geet recordings without domain-specific fine-tuning or model modification.

### Objectives

1. Evaluate pretrained HT-Demucs on three selected BAUL_GEET recordings.
2. Separate each recording into:
   - Vocals
   - No_Vocals (accompaniment/instrumental component)
3. Examine whether the pretrained model generalizes its source separation capability to Baul Geet recordings.
4. Maintain the standardized baseline experiment procedure established for EXP002.
5. Generate validated separation outputs for subsequent human qualitative evaluation.
6. Contribute evidence toward the investigation of pretrained source separation performance across Indian folk traditions.

### Hypothesis

The pretrained HT-Demucs model will produce usable two-stem vocal and accompaniment separation for the selected Baul Geet recordings, but some degree of vocal/instrumental leakage and separation artifacts may occur because the model was not specifically trained for Indian folk music.

### Experiment Scope

This experiment is strictly a baseline inference experiment.

No:
- fine-tuning,
- additional training,
- model architecture modification,
- instrument-specific separation,
- objective metric evaluation

will be performed in this notebook.

Objective evaluation will be performed separately during the project's objective evaluation phase.

### Experiment Configuration

- **Experiment ID:** EXP003
- **Tradition:** BAUL_GEET
- **Model:** HT-Demucs
- **Separation:** Two-stem
- **Input Recordings:** 3
- **Execution Environment:** Google Colab GPU
- **Evaluation After Experiment:** Human qualitative listening evaluation

# 2. Google Drive Setup

In [1]:
from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted successfully.")

Mounted at /content/drive
Google Drive mounted successfully.


# 3. Repository Configuration

In [2]:
from pathlib import Path
import sys
import os

# ============================================================================
# PROJECT CONFIGURATION
# ============================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/"
    "IKS_Music_Source_Separation_and_Instrument_Isolation"
)

INFRASTRUCTURE_ROOT = PROJECT_ROOT / "07_Infrastructure"

print("=" * 70)
print("Repository Configuration")
print("=" * 70)

print(f"Project Root   : {PROJECT_ROOT}")
print(f"Infrastructure : {INFRASTRUCTURE_ROOT}")

# Make project infrastructure importable
if str(INFRASTRUCTURE_ROOT) not in sys.path:
    sys.path.insert(0, str(INFRASTRUCTURE_ROOT))

print("\nRepository configuration loaded.")

Repository Configuration
Project Root   : /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation
Infrastructure : /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/07_Infrastructure

Repository configuration loaded.


# 4. Experiment Configuration

In [5]:
# ============================================================================
# EXPERIMENT CONFIGURATION
# ============================================================================

EXPERIMENT_ID = "EXP003"
NOTEBOOK_VERSION = "3.0"

MODEL_NAME = "HT-Demucs"
DEMUCS_MODEL = "htdemucs"

DATASET_NAME = "BAUL_GEET"

# Two-stem Demucs separation target
TWO_STEMS = "vocals"

# Exact recordings selected for EXP003
EXPECTED_RECORDINGS = [
    "BAUL_002.wav",
    "BAUL_016.wav",
    "BAUL_024.wav",
]

# ============================================================================
# DATASET PATH
# ============================================================================

DATASET_DIR = (
    PROJECT_ROOT
    / "03_Datasets"
    / DATASET_NAME
)

INPUT_AUDIO_DIR = (
    DATASET_DIR
    / "Audio"
    / "WAV"
)

# ============================================================================
# EXPERIMENT PATH
# ============================================================================

EXPERIMENT_DIR = (
    PROJECT_ROOT
    / "05_Experiments"
    / "Baseline_HT_Demucs"
    / EXPERIMENT_ID
)

OUTPUT_DIRECTORY = EXPERIMENT_DIR / "outputs"
LOG_DIRECTORY = EXPERIMENT_DIR / "logs"

# ============================================================================
# DISPLAY CONFIGURATION
# ============================================================================

print("=" * 70)
print("Experiment Configuration")
print("=" * 70)

print(f"Experiment ID       : {EXPERIMENT_ID}")
print(f"Notebook Version    : {NOTEBOOK_VERSION}")
print(f"Dataset             : {DATASET_NAME}")
print(f"Model               : {MODEL_NAME}")
print(f"Demucs Model        : {DEMUCS_MODEL}")
print(f"Separation          : Two-stem ({TWO_STEMS})")

print("\nExpected Recordings:")
for recording in EXPECTED_RECORDINGS:
    print(f"  - {recording}")

print("\nPaths:")
print(f"Dataset Directory   : {DATASET_DIR}")
print(f"Input Audio         : {INPUT_AUDIO_DIR}")
print(f"Experiment          : {EXPERIMENT_DIR}")
print(f"Outputs             : {OUTPUT_DIRECTORY}")
print(f"Logs                : {LOG_DIRECTORY}")

print("=" * 70)
print("Experiment configuration loaded successfully.")

Experiment Configuration
Experiment ID       : EXP003
Notebook Version    : 3.0
Dataset             : BAUL_GEET
Model               : HT-Demucs
Demucs Model        : htdemucs
Separation          : Two-stem (vocals)

Expected Recordings:
  - BAUL_002.wav
  - BAUL_016.wav
  - BAUL_024.wav

Paths:
Dataset Directory   : /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/03_Datasets/BAUL_GEET
Input Audio         : /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/03_Datasets/BAUL_GEET/Audio/WAV
Experiment          : /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/05_Experiments/Baseline_HT_Demucs/EXP003
Outputs             : /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/05_Experiments/Baseline_HT_Demucs/EXP003/outputs
Logs                : /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/05_Experiments/Baseline_HT_Demucs/EXP003/logs
Experiment configuration

# 5. Environment Validation

In [6]:
# ============================================================================
# ENVIRONMENT VALIDATION
# ============================================================================

import subprocess
import sys
import torch

print("=" * 70)
print("Environment Validation")
print("=" * 70)

# ---------------------------------------------------------------------------
# 1. Validate required directories
# ---------------------------------------------------------------------------

required_directories = {
    "Project Root": PROJECT_ROOT,
    "Infrastructure": INFRASTRUCTURE_ROOT,
    "Dataset Directory": DATASET_DIR,
    "Input Audio Directory": INPUT_AUDIO_DIR,
    "Experiment Directory": EXPERIMENT_DIR,
    "Output Directory": OUTPUT_DIRECTORY,
    "Log Directory": LOG_DIRECTORY,
}

print("\nChecking directories:")

for name, path in required_directories.items():

    if not path.exists():

        # Experiment/output/log directories are allowed to be created.
        if name in {
            "Experiment Directory",
            "Output Directory",
            "Log Directory",
        }:
            path.mkdir(parents=True, exist_ok=True)
            print(f"✓ Created {name}: {path}")

        else:
            raise FileNotFoundError(
                f"{name} does not exist:\n{path}"
            )

    else:
        print(f"✓ {name}: {path}")

# ---------------------------------------------------------------------------
# 2. Validate the exact recordings selected for EXP003
# ---------------------------------------------------------------------------

print("\nChecking selected recordings:")

expected_audio_files = []

for recording_name in EXPECTED_RECORDINGS:

    recording_path = INPUT_AUDIO_DIR / recording_name

    if not recording_path.exists():
        raise FileNotFoundError(
            f"Expected recording was not found:\n{recording_path}"
        )

    if recording_path.suffix.lower() != ".wav":
        raise ValueError(
            f"Expected a WAV file but found:\n{recording_path}"
        )

    expected_audio_files.append(recording_path)

    print(f"✓ {recording_name}")

print(
    f"\n✓ Validated all {len(expected_audio_files)} "
    "selected recordings."
)

# ---------------------------------------------------------------------------
# 3. Validate CUDA / GPU
# ---------------------------------------------------------------------------

print("\nChecking GPU environment:")

print(f"PyTorch Version : {torch.__version__}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available.\n"
        "Please enable a GPU runtime in Google Colab before continuing."
    )

gpu_name = torch.cuda.get_device_name(0)

print("✓ CUDA Available")
print(f"✓ GPU            : {gpu_name}")

# ---------------------------------------------------------------------------
# 4. Validate / install Demucs
# ---------------------------------------------------------------------------

print("\nChecking Demucs installation:")

try:

    import demucs

    demucs_version = demucs.__version__

    print("✓ Demucs is installed")
    print(f"✓ Demucs Version : {demucs_version}")

except ImportError:

    print("Demucs is not installed.")
    print("Installing Demucs...")

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "demucs",
        ],
        check=True,
    )

    import demucs

    demucs_version = demucs.__version__

    print("✓ Demucs installed successfully")
    print(f"✓ Demucs Version : {demucs_version}")

print("\n" + "=" * 70)
print("Environment validation passed successfully.")
print("=" * 70)

Environment Validation

Checking directories:
✓ Project Root: /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation
✓ Infrastructure: /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/07_Infrastructure
✓ Dataset Directory: /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/03_Datasets/BAUL_GEET
✓ Input Audio Directory: /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/03_Datasets/BAUL_GEET/Audio/WAV
✓ Experiment Directory: /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/05_Experiments/Baseline_HT_Demucs/EXP003
✓ Output Directory: /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/05_Experiments/Baseline_HT_Demucs/EXP003/outputs
✓ Log Directory: /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/05_Experiments/Baseline_HT_Demucs/EXP003/logs

Checking selected recordings:
✓ BAUL_002.wav
✓ BAUL_016.wav
✓ BAUL_024.w

# 6. HT-Demucs Execution

In [7]:
# ============================================================================
# HT-DEMUCS EXECUTION
# ============================================================================

import time
import subprocess
import datetime

print("=" * 70)
print("Running HT-Demucs Baseline Separation")
print("=" * 70)

print(f"Experiment : {EXPERIMENT_ID}")
print(f"Dataset    : {DATASET_NAME}")
print(f"Model      : {MODEL_NAME}")
print(f"Recordings : {len(expected_audio_files)}")

overall_start = time.time()
overall_start_dt = datetime.datetime.now()

results_log = []

for index, audio_file in enumerate(expected_audio_files, start=1):

    print(
        f"\n[{index}/{len(expected_audio_files)}] "
        f"Processing {audio_file.name}"
    )

    command = [
        sys.executable,
        "-m",
        "demucs",
        "--two-stems",
        TWO_STEMS,
        "-d",
        "cuda",
        "-o",
        str(OUTPUT_DIRECTORY),
        str(audio_file),
    ]

    print("\nCommand:")
    print(" ".join(command))

    start_time = time.time()

    result = subprocess.run(
        command,
        check=False,
    )

    processing_duration = time.time() - start_time

    if result.returncode != 0:

        status = "FAILED"

        print(
            f"✗ Demucs failed for {audio_file.name}"
        )

    else:

        status = "SUCCESS"

        print(
            f"✓ Completed in "
            f"{processing_duration:.2f} seconds"
        )

    results_log.append({
        "file": audio_file,
        "duration": processing_duration,
        "status": status,
        "returncode": result.returncode,
        "command": command,
    })

overall_end = time.time()
overall_end_dt = datetime.datetime.now()

total_duration = overall_end - overall_start

failed_recordings = [
    result
    for result in results_log
    if result["status"] != "SUCCESS"
]

if failed_recordings:

    print("\n" + "=" * 70)
    print("HT-Demucs execution FAILED")
    print("=" * 70)

    for result in failed_recordings:
        print(f"✗ {result['file'].name}")

    raise RuntimeError(
        "One or more expected recordings failed during "
        "HT-Demucs execution. Output processing will not continue."
    )

print("\n" + "=" * 70)
print("HT-Demucs execution completed successfully.")
print(f"Total processing time: {total_duration:.2f} seconds")
print("=" * 70)

Running HT-Demucs Baseline Separation
Experiment : EXP003
Dataset    : BAUL_GEET
Model      : HT-Demucs
Recordings : 3

[1/3] Processing BAUL_002.wav

Command:
/usr/bin/python3 -m demucs --two-stems vocals -d cuda -o /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/05_Experiments/Baseline_HT_Demucs/EXP003/outputs /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/03_Datasets/BAUL_GEET/Audio/WAV/BAUL_002.wav
✓ Completed in 29.56 seconds

[2/3] Processing BAUL_016.wav

Command:
/usr/bin/python3 -m demucs --two-stems vocals -d cuda -o /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/05_Experiments/Baseline_HT_Demucs/EXP003/outputs /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/03_Datasets/BAUL_GEET/Audio/WAV/BAUL_016.wav
✓ Completed in 34.87 seconds

[3/3] Processing BAUL_024.wav

Command:
/usr/bin/python3 -m demucs --two-stems vocals -d cuda -o /content/drive/MyDrive/IKS_Music_Sour

# 7. Output Organization

In [8]:
# ============================================================================
# OUTPUT ORGANIZATION
# ============================================================================

import shutil

print("=" * 70)
print("Organizing HT-Demucs Outputs")
print("=" * 70)

# ============================================================================
# DEMUCS TEMPORARY OUTPUT LOCATION
# ============================================================================

DEMUCS_OUTPUT_ROOT = OUTPUT_DIRECTORY / DEMUCS_MODEL

print(f"Temporary Demucs Output : {DEMUCS_OUTPUT_ROOT}")
print(f"Final Output Directory  : {OUTPUT_DIRECTORY}")

# Confirm that Demucs created its output directory
if not DEMUCS_OUTPUT_ROOT.exists():
    raise FileNotFoundError(
        "Demucs output directory was not found:\n"
        f"{DEMUCS_OUTPUT_ROOT}"
    )

# Store information about the final generated files
generated_files = []

# ============================================================================
# PROCESS EACH RECORDING
# ============================================================================

for result in results_log:

    if result["status"] != "SUCCESS":
        raise RuntimeError(
            f"Cannot organize output for failed recording:\n"
            f"{result['file'].name}"
        )

    recording_name = result["file"].stem

    print(f"\nProcessing output for: {recording_name}")

    # ------------------------------------------------------------------------
    # Locate Demucs-generated recording directory
    # ------------------------------------------------------------------------

    demucs_recording_dir = (
        DEMUCS_OUTPUT_ROOT
        / recording_name
    )

    if not demucs_recording_dir.exists():
        raise FileNotFoundError(
            "Expected Demucs recording directory was not found:\n"
            f"{demucs_recording_dir}"
        )

    # ------------------------------------------------------------------------
    # Locate original Demucs stem files
    # ------------------------------------------------------------------------

    vocals_source = (
        demucs_recording_dir
        / "vocals.wav"
    )

    no_vocals_source = (
        demucs_recording_dir
        / "no_vocals.wav"
    )

    if not vocals_source.exists():
        raise FileNotFoundError(
            "Demucs vocals output was not found:\n"
            f"{vocals_source}"
        )

    if not no_vocals_source.exists():
        raise FileNotFoundError(
            "Demucs no-vocals output was not found:\n"
            f"{no_vocals_source}"
        )

    # ------------------------------------------------------------------------
    # Create final recording directory
    # ------------------------------------------------------------------------

    final_recording_dir = (
        OUTPUT_DIRECTORY
        / recording_name
    )

    final_recording_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    # ------------------------------------------------------------------------
    # Define final standardized filenames
    # ------------------------------------------------------------------------

    vocals_target = (
        final_recording_dir
        / f"{recording_name}_Vocals.wav"
    )

    no_vocals_target = (
        final_recording_dir
        / f"{recording_name}_No_Vocals.wav"
    )

    # ------------------------------------------------------------------------
    # Prevent accidental overwrite of existing files
    # ------------------------------------------------------------------------

    if vocals_target.exists():
        raise FileExistsError(
            "Final vocals output already exists:\n"
            f"{vocals_target}"
        )

    if no_vocals_target.exists():
        raise FileExistsError(
            "Final no-vocals output already exists:\n"
            f"{no_vocals_target}"
        )

    # ------------------------------------------------------------------------
    # Move and rename the separated stems
    # ------------------------------------------------------------------------

    shutil.move(
        str(vocals_source),
        str(vocals_target),
    )

    shutil.move(
        str(no_vocals_source),
        str(no_vocals_target),
    )

    # ------------------------------------------------------------------------
    # Confirm final files exist
    # ------------------------------------------------------------------------

    if not vocals_target.exists():
        raise RuntimeError(
            "Vocals output was not successfully created:\n"
            f"{vocals_target}"
        )

    if not no_vocals_target.exists():
        raise RuntimeError(
            "No-vocals output was not successfully created:\n"
            f"{no_vocals_target}"
        )

    generated_files.append({
        "recording": recording_name,
        "vocals": vocals_target,
        "no_vocals": no_vocals_target,
    })

    print(f"✓ {recording_name}")
    print(f"  → {vocals_target.name}")
    print(f"  → {no_vocals_target.name}")

# ============================================================================
# REMOVE TEMPORARY DEMUCS DIRECTORY
# ============================================================================

if DEMUCS_OUTPUT_ROOT.exists():

    shutil.rmtree(DEMUCS_OUTPUT_ROOT)

    print(
        "\n✓ Removed temporary Demucs directory:"
        f"\n  {DEMUCS_OUTPUT_ROOT}"
    )

# ============================================================================
# FINAL ORGANIZATION CHECK
# ============================================================================

expected_output_count = len(EXPECTED_RECORDINGS) * 2

if len(generated_files) != len(EXPECTED_RECORDINGS):
    raise RuntimeError(
        "The number of successfully organized recordings does not match "
        "the number of expected recordings."
    )

print("\n" + "=" * 70)
print("Output organization completed successfully.")
print(f"Recordings organized : {len(generated_files)}")
print(f"Audio files created  : {expected_output_count}")
print("=" * 70)

Organizing HT-Demucs Outputs
Temporary Demucs Output : /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/05_Experiments/Baseline_HT_Demucs/EXP003/outputs/htdemucs
Final Output Directory  : /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/05_Experiments/Baseline_HT_Demucs/EXP003/outputs

Processing output for: BAUL_002
✓ BAUL_002
  → BAUL_002_Vocals.wav
  → BAUL_002_No_Vocals.wav

Processing output for: BAUL_016
✓ BAUL_016
  → BAUL_016_Vocals.wav
  → BAUL_016_No_Vocals.wav

Processing output for: BAUL_024
✓ BAUL_024
  → BAUL_024_Vocals.wav
  → BAUL_024_No_Vocals.wav

✓ Removed temporary Demucs directory:
  /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/05_Experiments/Baseline_HT_Demucs/EXP003/outputs/htdemucs

Output organization completed successfully.
Recordings organized : 3
Audio files created  : 6


# 8. Output Structure and Audio Integrity Validation

In [9]:
# ============================================================================
# OUTPUT STRUCTURE AND AUDIO INTEGRITY VALIDATION
# ============================================================================

import soundfile as sf

print("=" * 70)
print("Validating Final Output Structure and Audio Integrity")
print("=" * 70)

audio_validation = []

expected_output_count = len(EXPECTED_RECORDINGS) * 2

# ============================================================================
# VALIDATE EACH RECORDING
# ============================================================================

for recording_name in EXPECTED_RECORDINGS:

    recording_stem = Path(recording_name).stem

    recording_output_dir = (
        OUTPUT_DIRECTORY
        / recording_stem
    )

    print(f"\nChecking: {recording_stem}")

    # ------------------------------------------------------------------------
    # Validate recording output directory
    # ------------------------------------------------------------------------

    if not recording_output_dir.exists():
        raise FileNotFoundError(
            "Output directory is missing:\n"
            f"{recording_output_dir}"
        )

    # ------------------------------------------------------------------------
    # Define the two expected output files
    # ------------------------------------------------------------------------

    vocals_file = (
        recording_output_dir
        / f"{recording_stem}_Vocals.wav"
    )

    no_vocals_file = (
        recording_output_dir
        / f"{recording_stem}_No_Vocals.wav"
    )

    expected_files = [
        vocals_file,
        no_vocals_file,
    ]

    # ------------------------------------------------------------------------
    # Validate exact file count
    # ------------------------------------------------------------------------

    actual_wav_files = sorted(
        recording_output_dir.glob("*.wav")
    )

    if len(actual_wav_files) != 2:
        raise RuntimeError(
            f"Expected exactly 2 WAV files in:\n"
            f"{recording_output_dir}\n"
            f"Found: {len(actual_wav_files)}"
        )

    # ------------------------------------------------------------------------
    # Validate each expected audio file
    # ------------------------------------------------------------------------

    for audio_file in expected_files:

        if not audio_file.exists():
            raise FileNotFoundError(
                "Expected output file is missing:\n"
                f"{audio_file}"
            )

        # ------------------------------------------------------------
        # File size validation
        # ------------------------------------------------------------

        file_size = audio_file.stat().st_size

        if file_size <= 0:
            raise ValueError(
                f"Output file is empty:\n{audio_file}"
            )

        # ------------------------------------------------------------
        # Audio readability validation
        # ------------------------------------------------------------

        try:
            data, sample_rate = sf.read(audio_file)

        except Exception as error:
            raise RuntimeError(
                f"Unable to read audio file:\n"
                f"{audio_file}\n\n"
                f"Error: {error}"
            ) from error

        # ------------------------------------------------------------
        # Sample rate validation
        # ------------------------------------------------------------

        if sample_rate <= 0:
            raise ValueError(
                f"Invalid sample rate:\n"
                f"{audio_file}"
            )

        # ------------------------------------------------------------
        # Audio data validation
        # ------------------------------------------------------------

        if len(data) == 0:
            raise ValueError(
                f"Audio file contains no samples:\n"
                f"{audio_file}"
            )

        # ------------------------------------------------------------
        # Duration calculation
        # ------------------------------------------------------------

        duration = len(data) / sample_rate

        if duration <= 0:
            raise ValueError(
                f"Invalid duration:\n"
                f"{audio_file}"
            )

        # ------------------------------------------------------------
        # Channel calculation
        # ------------------------------------------------------------

        channels = (
            data.shape[1]
            if data.ndim > 1
            else 1
        )

        # ------------------------------------------------------------
        # Store validation information
        # ------------------------------------------------------------

        audio_validation.append({
            "File": audio_file.name,
            "Sample Rate": sample_rate,
            "Channels": channels,
            "Duration": round(duration, 2),
            "Size Bytes": file_size,
        })

        print(
            f"  ✓ {audio_file.name}"
            f" | {sample_rate} Hz"
            f" | {channels} ch"
            f" | {duration:.2f} s"
            f" | {file_size:,} bytes"
        )

# ============================================================================
# VALIDATE TOTAL OUTPUT COUNT
# ============================================================================

if len(audio_validation) != expected_output_count:
    raise RuntimeError(
        f"Expected {expected_output_count} separated audio files, "
        f"but validated {len(audio_validation)}."
    )

# ============================================================================
# VERIFY TEMPORARY DEMUCS DIRECTORY WAS REMOVED
# ============================================================================

temporary_demucs_directory = (
    OUTPUT_DIRECTORY
    / DEMUCS_MODEL
)

if temporary_demucs_directory.exists():
    raise RuntimeError(
        "Temporary Demucs directory still exists:\n"
        f"{temporary_demucs_directory}"
    )

# ============================================================================
# FINAL VALIDATION SUMMARY
# ============================================================================

print("\n" + "=" * 70)
print("Audio Integrity Validation Completed")
print("=" * 70)

print(
    f"Expected separated files : {expected_output_count}"
)

print(
    f"Validated separated files: {len(audio_validation)}"
)

print("✓ All expected recording directories exist.")
print("✓ Each recording contains exactly two WAV files.")
print("✓ All expected output filenames are correct.")
print("✓ All audio files are non-empty.")
print("✓ All audio files are readable.")
print("✓ All sample rates are valid.")
print("✓ All durations are positive.")
print("✓ Temporary Demucs directory has been removed.")

print("=" * 70)
print("All audio integrity checks passed successfully.")
print("=" * 70)

Validating Final Output Structure and Audio Integrity

Checking: BAUL_002
  ✓ BAUL_002_Vocals.wav | 44100 Hz | 2 ch | 336.54 s | 59,365,164 bytes
  ✓ BAUL_002_No_Vocals.wav | 44100 Hz | 2 ch | 336.54 s | 59,365,164 bytes

Checking: BAUL_016
  ✓ BAUL_016_Vocals.wav | 44100 Hz | 2 ch | 550.42 s | 97,093,420 bytes
  ✓ BAUL_016_No_Vocals.wav | 44100 Hz | 2 ch | 550.42 s | 97,093,420 bytes

Checking: BAUL_024
  ✓ BAUL_024_Vocals.wav | 44100 Hz | 2 ch | 177.15 s | 31,248,428 bytes
  ✓ BAUL_024_No_Vocals.wav | 44100 Hz | 2 ch | 177.15 s | 31,248,428 bytes

Audio Integrity Validation Completed
Expected separated files : 6
Validated separated files: 6
✓ All expected recording directories exist.
✓ Each recording contains exactly two WAV files.
✓ All expected output filenames are correct.
✓ All audio files are non-empty.
✓ All audio files are readable.
✓ All sample rates are valid.
✓ All durations are positive.
✓ Temporary Demucs directory has been removed.
All audio integrity checks passed succe

# 9. Execution Log Generation

In [10]:
# ============================================================================
# EXECUTION LOG GENERATION
# ============================================================================

print("=" * 70)
print("Generating Experiment Execution Log")
print("=" * 70)

# ============================================================================
# LOG FILE PATH
# ============================================================================

log_file_path = (
    LOG_DIRECTORY
    / "execution_log.txt"
)

# ============================================================================
# DETERMINE EXPERIMENT STATUS
# ============================================================================

successful_recordings = [
    result
    for result in results_log
    if result["status"] == "SUCCESS"
]

failed_recordings = [
    result
    for result in results_log
    if result["status"] != "SUCCESS"
]

experiment_status = (
    "SUCCESS"
    if (
        len(failed_recordings) == 0
        and len(audio_validation) == expected_output_count
    )
    else "FAILED"
)

# ============================================================================
# BUILD EXECUTION LOG
# ============================================================================

log_content = ""

log_content += "=" * 70 + "\n"
log_content += "IKS INTERNSHIP EXPERIMENT EXECUTION LOG\n"
log_content += "=" * 70 + "\n\n"

# ---------------------------------------------------------------------------
# Experiment Information
# ---------------------------------------------------------------------------

log_content += "=" * 70 + "\n"
log_content += "Experiment Information\n"
log_content += "=" * 70 + "\n"

log_content += f"Experiment ID          : {EXPERIMENT_ID}\n"
log_content += f"Notebook Version       : {NOTEBOOK_VERSION}\n"
log_content += f"Dataset                : {DATASET_NAME}\n"
log_content += f"Model                  : {MODEL_NAME}\n"
log_content += f"Demucs Model           : {DEMUCS_MODEL}\n"
log_content += f"Separation Type        : Two-stem (Vocals / No_Vocals)\n"
log_content += f"Number of Recordings   : {len(EXPECTED_RECORDINGS)}\n\n"

# ---------------------------------------------------------------------------
# Execution Timing
# ---------------------------------------------------------------------------

log_content += "=" * 70 + "\n"
log_content += "Execution Timing\n"
log_content += "=" * 70 + "\n"

log_content += (
    f"Execution Start        : "
    f"{overall_start_dt.strftime('%Y-%m-%d %H:%M:%S')}\n"
)

log_content += (
    f"Execution End          : "
    f"{overall_end_dt.strftime('%Y-%m-%d %H:%M:%S')}\n"
)

log_content += (
    f"Total Processing Time  : "
    f"{total_duration:.2f} seconds\n\n"
)

# ---------------------------------------------------------------------------
# Environment
# ---------------------------------------------------------------------------

log_content += "=" * 70 + "\n"
log_content += "Environment\n"
log_content += "=" * 70 + "\n"

log_content += f"PyTorch Version        : {torch.__version__}\n"
log_content += f"GPU                    : {gpu_name}\n"
log_content += f"Demucs Version         : {demucs_version}\n\n"

# ---------------------------------------------------------------------------
# Repository Paths
# ---------------------------------------------------------------------------

log_content += "=" * 70 + "\n"
log_content += "Repository Paths\n"
log_content += "=" * 70 + "\n"

log_content += f"Project Root           : {PROJECT_ROOT}\n"
log_content += f"Infrastructure         : {INFRASTRUCTURE_ROOT}\n"
log_content += f"Dataset Directory      : {DATASET_DIR}\n"
log_content += f"Input Audio Directory  : {INPUT_AUDIO_DIR}\n"
log_content += f"Experiment Directory   : {EXPERIMENT_DIR}\n"
log_content += f"Output Directory       : {OUTPUT_DIRECTORY}\n"
log_content += f"Logs Directory         : {LOG_DIRECTORY}\n\n"

# ---------------------------------------------------------------------------
# Input Recordings
# ---------------------------------------------------------------------------

log_content += "=" * 70 + "\n"
log_content += "Input Recordings\n"
log_content += "=" * 70 + "\n"

for recording in EXPECTED_RECORDINGS:
    log_content += f"- {recording}\n"

log_content += "\n"

# ---------------------------------------------------------------------------
# Processing Details
# ---------------------------------------------------------------------------

log_content += "=" * 70 + "\n"
log_content += "Processing Details\n"
log_content += "=" * 70 + "\n"

for result in results_log:

    log_content += f"\nRecording             : {result['file'].name}\n"
    log_content += f"Status                : {result['status']}\n"
    log_content += (
        f"Processing Time       : "
        f"{result['duration']:.2f} seconds\n"
    )
    log_content += f"Return Code           : {result['returncode']}\n"
    log_content += (
        f"Command               : "
        f"{' '.join(result['command'])}\n"
    )

# ---------------------------------------------------------------------------
# Generated Outputs
# ---------------------------------------------------------------------------

log_content += "\n"
log_content += "=" * 70 + "\n"
log_content += "Generated Outputs\n"
log_content += "=" * 70 + "\n"

for result in generated_files:

    log_content += f"\n{result['recording']}\n"
    log_content += f"  - {result['vocals'].name}\n"
    log_content += f"  - {result['no_vocals'].name}\n"

# ---------------------------------------------------------------------------
# Audio Validation
# ---------------------------------------------------------------------------

log_content += "\n"
log_content += "=" * 70 + "\n"
log_content += "Audio Integrity Validation\n"
log_content += "=" * 70 + "\n"

for item in audio_validation:

    log_content += (
        f"\nFile                  : {item['File']}\n"
        f"Sample Rate           : {item['Sample Rate']} Hz\n"
        f"Channels              : {item['Channels']}\n"
        f"Duration              : {item['Duration']} seconds\n"
        f"File Size             : {item['Size Bytes']} bytes\n"
    )

# ---------------------------------------------------------------------------
# Final Status
# ---------------------------------------------------------------------------

log_content += "\n"
log_content += "=" * 70 + "\n"
log_content += "Final Experiment Status\n"
log_content += "=" * 70 + "\n"

log_content += f"Experiment Status      : {experiment_status}\n"
log_content += (
    f"Recordings Processed   : "
    f"{len(successful_recordings)} / "
    f"{len(EXPECTED_RECORDINGS)}\n"
)
log_content += (
    f"Separated Audio Files  : "
    f"{len(audio_validation)} / "
    f"{expected_output_count}\n"
)
log_content += "Audio Validation       : PASSED\n"
log_content += "Manual Evaluation      : PENDING\n"
log_content += "Objective Evaluation   : PENDING\n"

log_content += "\n"
log_content += "=" * 70 + "\n"
log_content += "Next Step: Manual listening evaluation of all Vocals and No_Vocals outputs.\n"
log_content += "=" * 70 + "\n"

# ============================================================================
# WRITE LOG FILE
# ============================================================================

with open(
    log_file_path,
    "w",
    encoding="utf-8",
) as log_file:

    log_file.write(log_content)

# ============================================================================
# VERIFY LOG FILE
# ============================================================================

if not log_file_path.exists():
    raise RuntimeError(
        f"Execution log was not created:\n{log_file_path}"
    )

if log_file_path.stat().st_size == 0:
    raise RuntimeError(
        f"Execution log is empty:\n{log_file_path}"
    )

print(f"✓ Execution log saved to:")
print(f"  {log_file_path}")

print(f"\n✓ Log file size: {log_file_path.stat().st_size:,} bytes")
print(f"✓ Experiment status: {experiment_status}")

print("\n" + "=" * 70)
print("Execution log generation completed successfully.")
print("=" * 70)

Generating Experiment Execution Log
✓ Execution log saved to:
  /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/05_Experiments/Baseline_HT_Demucs/EXP003/logs/execution_log.txt

✓ Log file size: 6,161 bytes
✓ Experiment status: SUCCESS

Execution log generation completed successfully.


# 10. Final Experiment Summary

In [11]:
# ============================================================================
# FINAL EXPERIMENT SUMMARY
# ============================================================================

print("=" * 70)
print("EXP003 FINAL EXPERIMENT SUMMARY")
print("=" * 70)

print(f"Experiment ID       : {EXPERIMENT_ID}")
print(f"Dataset             : {DATASET_NAME}")
print(f"Model               : {MODEL_NAME}")
print(f"Demucs Model        : {DEMUCS_MODEL}")
print(f"Separation Type     : Two-stem (Vocals / No_Vocals)")

print("\nSelected Recordings:")
for recording in EXPECTED_RECORDINGS:
    print(f"  ✓ {recording}")

print("\nProcessing:")
print(f"  Recordings Processed : {len(successful_recordings)} / {len(EXPECTED_RECORDINGS)}")
print(f"  Audio Files          : {len(audio_validation)} / {expected_output_count}")
print(f"  Total Time           : {total_duration:.2f} seconds")

print("\nValidation:")
print("  ✓ Output structure validated")
print("  ✓ Audio integrity validated")
print("  ✓ All output files readable")
print("  ✓ All expected stems present")

print("\nExperiment Status:")
print(f"  {experiment_status}")

print("\nExecution Log:")
print(f"  {log_file_path}")

print("\nNext Steps:")
print("  1. Manually listen to the six separated outputs.")
print("  2. Record observations for each of the three recordings.")
print("  3. Prepare observations.md and conclusions.md.")
print("  4. Update the Master Evaluation Checklist.")
print("  5. Perform objective evaluation separately.")

print("=" * 70)

if experiment_status != "SUCCESS":
    raise RuntimeError(
        "EXP003 did not complete successfully. "
        "Review the execution log before proceeding."
    )

print("\n✓ EXP003 baseline experiment completed successfully.")
print("✓ Outputs are ready for manual qualitative evaluation.")

EXP003 FINAL EXPERIMENT SUMMARY
Experiment ID       : EXP003
Dataset             : BAUL_GEET
Model               : HT-Demucs
Demucs Model        : htdemucs
Separation Type     : Two-stem (Vocals / No_Vocals)

Selected Recordings:
  ✓ BAUL_002.wav
  ✓ BAUL_016.wav
  ✓ BAUL_024.wav

Processing:
  Recordings Processed : 3 / 3
  Audio Files          : 6 / 6
  Total Time           : 78.55 seconds

Validation:
  ✓ Output structure validated
  ✓ Audio integrity validated
  ✓ All output files readable
  ✓ All expected stems present

Experiment Status:
  SUCCESS

Execution Log:
  /content/drive/MyDrive/IKS_Music_Source_Separation_and_Instrument_Isolation/05_Experiments/Baseline_HT_Demucs/EXP003/logs/execution_log.txt

Next Steps:
  1. Manually listen to the six separated outputs.
  2. Record observations for each of the three recordings.
  3. Prepare observations.md and conclusions.md.
  4. Update the Master Evaluation Checklist.
  5. Perform objective evaluation separately.

✓ EXP003 baseline 